In [1]:
### Program to calculate the CHSH values. 
### This program is drafted by Deepseek, and later checked and edited by the authors. 

import numpy as np
import math

def pauli_matrices():
    """Return Pauli matrices"""
    sigma_x = np.array([[0, 1], [1, 0]])
    sigma_y = np.array([[0, -1j], [1j, 0]])
    sigma_z = np.array([[1, 0], [0, -1]])
    return sigma_x, sigma_y, sigma_z

def measurement_operator(theta, phi):
    """
    Create measurement operator for given Bloch sphere angles
    
    Args:
        theta: polar angle (0 to pi)
        phi: azimuthal angle (0 to 2pi)
    
    Returns:
        M_plus: projector for +1 outcome
        M_minus: projector for -1 outcome
    """
    # Bloch vector components
    nx = np.sin(theta) * np.cos(phi)
    ny = np.sin(theta) * np.sin(phi)
    nz = np.cos(theta)
    
    sigma_x, sigma_y, sigma_z = pauli_matrices()
    
    # Measurement operator M = n·σ
    M = nx * sigma_x + ny * sigma_y + nz * sigma_z
    
    # Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(M)
    
    # +1 outcome projector (larger eigenvalue)
    if eigenvalues[1] > eigenvalues[0]:
        M_plus = np.outer(eigenvectors[:, 1], eigenvectors[:, 1].conj())
        M_minus = np.outer(eigenvectors[:, 0], eigenvectors[:, 0].conj())
    else:
        M_plus = np.outer(eigenvectors[:, 0], eigenvectors[:, 0].conj())
        M_minus = np.outer(eigenvectors[:, 1], eigenvectors[:, 1].conj())
    print (M_plus)
    
    return M_plus, M_minus

def calculate_probabilities(theta_a, phi_a, theta_b, phi_b, state):
    """
    Calculate probabilities for all outcome pairs
    
    Returns:
        p_plus_plus: P(+1,+1)
        p_plus_minus: P(+1,-1) 
        p_minus_plus: P(-1,+1)
        p_minus_minus: P(-1,-1)
    """
    # Get measurement operators
    A_plus, A_minus = measurement_operator(theta_a, phi_a)
    B_plus, B_minus = measurement_operator(theta_b, phi_b)
    
    # Projectors for joint measurements
    P_plus_plus = np.kron(A_plus, B_plus)
    P_plus_minus = np.kron(A_plus, B_minus)
    P_minus_plus = np.kron(A_minus, B_plus)
    P_minus_minus = np.kron(A_minus, B_minus)
    
    # Calculate probabilities
    p_plus_plus = (state.conj().T @ P_plus_plus @ state).real
    p_plus_minus = (state.conj().T @ P_plus_minus @ state).real
    p_minus_plus = (state.conj().T @ P_minus_plus @ state).real
    p_minus_minus = (state.conj().T @ P_minus_minus @ state).real
    
    probabilities = {
        'P(+,+)': p_plus_plus,
        'P(+,-)': p_plus_minus,
        'P(-,+)': p_minus_plus,
        'P(-,-)': p_minus_minus
    }
    
    return probabilities

def correlation_with_probabilities(theta_a, phi_a, theta_b, phi_b, state):
    """
    Calculate correlation and probabilities
    """
    probabilities = calculate_probabilities(theta_a, phi_a, theta_b, phi_b, state)
    
    # Correlation from probabilities: E = p_pp + p_mm - p_pm - p_mp
    correlation = probabilities['P(+,+)'] + probabilities['P(-,-)'] - probabilities['P(+,-)'] - probabilities['P(-,+)']
    
    return correlation, probabilities

def calculate_chsh_with_probabilities(theta_a, phi_a, theta_ap, phi_ap, 
                                    theta_b, phi_b, theta_bp, phi_bp):
    """
    Calculate CHSH value with probabilities for all measurements
    """
    entangled_state = np.array([1, 0, 0, 1]) / np.sqrt(2)
    entangled_state = np.array([1, 0, 0, 0])
    entangled_state = np.array([1/2, 1/2, 1/2, 1/2])
    
    # Calculate correlations and probabilities for all four settings
    E_ab, prob_ab = correlation_with_probabilities(theta_a, phi_a, theta_b, phi_b, entangled_state)
    E_abp, prob_abp = correlation_with_probabilities(theta_a, phi_a, theta_bp, phi_bp, entangled_state)
    E_apb, prob_apb = correlation_with_probabilities(theta_ap, phi_ap, theta_b, phi_b, entangled_state)
    E_apbp, prob_apbp = correlation_with_probabilities(theta_ap, phi_ap, theta_bp, phi_bp, entangled_state)
    
    S1 = abs(E_ab + E_abp + E_apb - E_apbp)
    S2 = abs(E_ab + E_abp - E_apb + E_apbp)
    S3 = abs(E_ab - E_abp + E_apb + E_apbp)
    S4 = abs(-E_ab + E_abp + E_apb + E_apbp)
    S  = max(S1, S2, S3, S4)
    
    
    correlations = {
        'E(a,b)': E_ab, 'prob(a,b)': prob_ab,
        'E(a,b\')': E_abp, 'prob(a,b\')': prob_abp,
        'E(a\',b)': E_apb, 'prob(a\',b)': prob_apb,
        'E(a\',b\')': E_apbp, 'prob(a\',b\')': prob_apbp
    }
    
    return S, correlations

def main():
    """Main function to run CHSH calculation"""
    print("CHSH Value Calculator with Probabilities")
    print("========================================")
    
    # Get user input for measurement angles
    print("\nEnter measurement angles (in radians):")
    
    # Alice's measurements
    theta_a = float(str(math.pi*25/180))
    phi_a = float(str(math.pi*163/180))
    
    theta_ap = float(str(math.pi*88/180))
    phi_ap = float(str(math.pi*300/180))
    
    # Bob's measurements  
    theta_b = float(str(math.pi*8/180))
    phi_b = float(str(math.pi*270/180))
    
    theta_bp = float(str(math.pi*100/180))
    phi_bp = float(str(math.pi*24/180))
    
    
    # Calculate CHSH value with probabilities
    S, correlations = calculate_chsh_with_probabilities(theta_a, phi_a, theta_ap, phi_ap, 
                                   theta_b, phi_b, theta_bp, phi_bp)
    
    # Display results
    print(f"\n" + "="*60)
    print("RESULTS")
    print("="*60)
    
    # Display correlations and probabilities for each measurement setting
    settings = [
        ('(a,b)', 'E(a,b)', 'prob(a,b)'),
        ('(a,b\')', 'E(a,b\')', 'prob(a,b\')'),
        ('(a\',b)', 'E(a\',b)', 'prob(a\',b)'),
        ('(a\',b\')', 'E(a\',b\')', 'prob(a\',b\')')
    ]
    
    for setting_name, E_key, prob_key in settings:
        print(f"\nMeasurement {setting_name}:")
        print(f"  Correlation = {correlations[E_key]:.6f}")
        print(f"  Probabilities:")
        probabilities = correlations[prob_key]
        for outcome, prob in probabilities.items():
            print(f"    {outcome} = {prob:.6f}")
    
    print(f"\n" + "="*40)
    print(f"CHSH VALUE: S = {S:.6f}")
    print("="*40)
    
    # Probability verification
    print(f"\nProbability Verification:")
    for setting_name, E_key, prob_key in settings:
        probabilities = correlations[prob_key]
        total = sum(probabilities.values())
        print(f"  {setting_name}: total = {total:.6f} {'✓' if abs(total-1.0) < 1e-10 else '✗'}")
    
    # Interpretation
    print(f"\nInterpretation:")
    if S > 2:
        print("✅ Violates CHSH inequality! Quantum entanglement detected.")
        violation_amount = S - 2
        print(f"   Violation amount: {violation_amount:.4f}")
        if S > 2.8:
            print("🎯 Close to quantum maximum 2√2 ≈ 2.828")
    else:
        print("❌ Does not violate CHSH inequality")




if __name__ == "__main__":
    main()
    

CHSH Value Calculator with Probabilities

Enter measurement angles (in radians):
[[ 0.95315389+0.00000000e+00j -0.20207593-6.17808108e-02j]
 [-0.20207593+6.17808108e-02j  0.04684611+7.10865738e-19j]]
[[ 9.95134034e-01+0.j         -1.38439822e-17+0.06958655j]
 [-1.38439822e-17-0.06958655j  4.86596563e-03+0.j        ]]
[[ 0.95315389+0.00000000e+00j -0.20207593-6.17808108e-02j]
 [-0.20207593+6.17808108e-02j  0.04684611+7.10865738e-19j]]
[[0.41317591+0.00000000e+00j 0.44983332-2.00278700e-01j]
 [0.44983332+2.00278700e-01j 0.58682409+2.30653581e-18j]]
[[0.51744975+0.00000000e+00j 0.24984771+4.32748922e-01j]
 [0.24984771-4.32748922e-01j 0.48255025-9.47169042e-19j]]
[[ 9.95134034e-01+0.j         -1.38439822e-17+0.06958655j]
 [-1.38439822e-17-0.06958655j  4.86596563e-03+0.j        ]]
[[0.51744975+0.00000000e+00j 0.24984771+4.32748922e-01j]
 [0.24984771-4.32748922e-01j 0.48255025-9.47169042e-19j]]
[[0.41317591+0.00000000e+00j 0.44983332-2.00278700e-01j]
 [0.44983332+2.00278700e-01j 0.58682409+2